<a href="https://colab.research.google.com/github/sophia-yang424/pneumonia_class/blob/dev/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
!pip install colab-xterm
%load_ext colabxterm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 2.1 MB/s eta 0:00:00


In [ ]:
!pip install numpy
!pip install datasets
!pip install transformers
!pip install sklearn
!pip install gradio

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
# --- 1. Load dataset (same one as linear probing exercise) ---
dataset_name = "mmenendezg/pneumonia_x_ray"
ds = load_dataset(dataset_name)

# Subsample for CPU time budget — same scale as the linear probing exercise
train_ds = ds["train"].shuffle(seed=42).select(range(800))
test_ds = ds["test"].shuffle(seed=42).select(range(200))

num_labels = 2  # normal vs pneumonia


README.md:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  110MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.6MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.2MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4187 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1045 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/624 [00:00<?, ? examples/s]

In [ ]:
print(ds["train"].features["label"])

ClassLabel(names=['normal', 'pneumonia'])


In [ ]:
# --- 2. Load processor + model ---
model_name = "microsoft/resnet-18"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)

# No freezing here — this is the fine-tuning exercise, so every parameter
# (backbone + head) keeps its default requires_grad=True and gets updated
# during training.


preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.5k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
# --- 3. Preprocessing function ---
def preprocess(examples):
    inputs = processor([img.convert("RGB") for img in examples["image"]], return_tensors="pt")
    examples["pixel_values"] = inputs["pixel_values"]
    return examples

train_processed = train_ds.map(preprocess, batched=True)
test_processed = test_ds.map(preprocess, batched=True)


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
# --- 4. Evaluation metric ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

In [ ]:
# --- 5. Training arguments ---
training_args = TrainingArguments(
    output_dir="./fine_tuned_pneumonia_model",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    learning_rate=3e-5,
    eval_strategy="epoch",
    logging_steps=10,
)

In [ ]:
# --- 6. Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=test_processed,
    compute_metrics=compute_metrics,
)
trainer.train()
# train() wraps these 3: backward, step, zero_grad aka does backprop for however many epochs we specified in traning_args

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


In [ ]:
# --- 7. Evaluate ---
results = trainer.evaluate()
print(results)

# --- 8. Save fine-tuned model ---
trainer.save_model("./fine_tuned_pneumonia_model")

In [ ]:
import gradio as gr
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor

model = AutoModelForImageClassification.from_pretrained("./fine_tuned_pneumonia_model")
processor = AutoImageProcessor.from_pretrained("./fine_tuned_pneumonia_model")
model.eval()

def predict(image):
    inputs = processor(images=image.convert("RGB"), return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax(-1).item()
    return "Pneumonia" if pred == 1 else "Normal"

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Pneumonia X-Ray Classifier (Fine-Tuned ResNet-18)",
)

demo.launch()